> 🔑 **BẢN LỜI GIẢI** — chỉ mở khi đã thử tự làm và đọc gợi ý.

# Lab 1 — Viết training loop từ đầu với PyTorch

**Mục tiêu:** train một CNN phân loại ảnh CIFAR-10 bằng PyTorch thuần (không Lightning, không `Trainer`), đạt **val accuracy ≥ 70%**, và train tiếp được từ checkpoint.

**Sau lab này bạn sẽ hiểu:**
1. `Dataset`, `DataLoader`, transform và data augmentation hoạt động thế nào
2. Cấu trúc một CNN cơ bản (Conv → BatchNorm → ReLU → Pool)
3. 5 bước của một training step: `zero_grad → forward → loss → backward → step`
4. Khác nhau giữa `model.train()` và `model.eval()`, và vì sao cần `torch.no_grad()`
5. Lưu và nạp checkpoint để train tiếp

**Cách làm:**
- Mở notebook trên **Google Colab** → menu *Runtime → Change runtime type → T4 GPU*.
- Chạy lần lượt từ trên xuống. Chỗ nào có `# TODO` là chỗ bạn viết code.
- Sau mỗi phần có một ô **✅ Kiểm tra**. Chạy nó: in ra ✅ là đúng, báo lỗi thì đọc thông báo và sửa.
- Bí quá thì mở phần **Gợi ý** ngay dưới TODO trước. Vẫn bí mới xem file lời giải.

⏱️ Thời gian dự kiến: 2–3 giờ (trong đó train khoảng 10 phút trên GPU).

## Phần 0 — Chuẩn bị

In [ ]:
import os, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| device:", device)
if device == "cpu":
    print("⚠️ Đang chạy CPU. Vào Runtime → Change runtime type → chọn GPU để train nhanh hơn ~20 lần.")

## Phần 1 — Dữ liệu

CIFAR-10 có 60.000 ảnh màu 32×32, chia 10 lớp (máy bay, ô tô, chim, mèo...). 50.000 ảnh train, 10.000 ảnh test.

Ta chia 50.000 ảnh train thành **45.000 train** + **5.000 validation**. Tập test chỉ dùng **một lần duy nhất** ở cuối, để kết quả không bị "nhìn trộm".

### Transform là gì?
Ảnh gốc là ảnh PIL, giá trị pixel 0–255. Model cần tensor số thực đã chuẩn hóa. Transform là chuỗi bước biến đổi ảnh trước khi đưa vào model:

| Bước | Tác dụng |
|---|---|
| `T.RandomCrop(32, padding=4)` | Đệm 4 pixel quanh ảnh rồi cắt ngẫu nhiên 32×32 → model quen với vật thể lệch vị trí |
| `T.RandomHorizontalFlip()` | Lật ngang ngẫu nhiên 50% → ảnh con mèo quay trái hay phải đều là mèo |
| `T.ToTensor()` | PIL → tensor `[3, 32, 32]`, giá trị về 0–1 |
| `T.Normalize(MEAN, STD)` | Trừ trung bình, chia độ lệch chuẩn từng kênh màu → giá trị quanh 0, train ổn định hơn |

Hai bước đầu là **data augmentation**: chỉ dùng khi **train**. Khi đánh giá (val/test) ta muốn ảnh giữ nguyên, nên chỉ `ToTensor` + `Normalize`.

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)   # trung bình từng kênh R, G, B của CIFAR-10
STD  = (0.2470, 0.2435, 0.2616)   # độ lệch chuẩn từng kênh

train_tf = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

test_tf = T.Compose([
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])

<details><summary>💡 Gợi ý</summary>

Dùng `T.Compose([...])` để nối các bước thành một danh sách:
```python
train_tf = T.Compose([
    T.RandomCrop(...),
    ...
])
```
</details>

Ô dưới tải dữ liệu (đã viết sẵn). Chú ý mẹo: ta tạo **hai** bản của tập train, một bản dùng `train_tf`, một bản dùng `test_tf`, rồi chia chỉ số. Nếu chỉ tạo một bản rồi `random_split`, tập val sẽ bị dính augmentation, một lỗi rất hay gặp.

In [ ]:
DATA_DIR = "./data"
full_train_aug   = torchvision.datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=train_tf)
full_train_plain = torchvision.datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=test_tf)
test_set         = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=test_tf)

g = torch.Generator().manual_seed(42)
perm = torch.randperm(len(full_train_aug), generator=g).tolist()
n_val = 5000
train_set = Subset(full_train_aug,   perm[n_val:])
val_set   = Subset(full_train_plain, perm[:n_val])

CLASSES = full_train_aug.classes
print(f"train: {len(train_set)} | val: {len(val_set)} | test: {len(test_set)}")
print("Các lớp:", CLASSES)

### DataLoader
`DataLoader` gom ảnh thành từng **batch** (lô), xáo trộn và tải song song.
- `batch_size=128`: mỗi bước train xử lý 128 ảnh.
- `shuffle=True` **chỉ cho train**: mỗi epoch xáo thứ tự để model không học thuộc thứ tự. Val/test không cần xáo.
- `num_workers=2`: dùng 2 tiến trình phụ để đọc ảnh song song.

In [ ]:
BATCH_SIZE = 128

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# ✅ Kiểm tra Phần 1
assert train_tf is not None and test_tf is not None, "Chưa làm TODO 1.1 / 1.2"
assert train_loader is not None and val_loader is not None and test_loader is not None, "Chưa làm TODO 1.3"
x, y = next(iter(train_loader))
assert x.shape == (BATCH_SIZE, 3, 32, 32), f"Batch ảnh phải có shape (128, 3, 32, 32), đang là {tuple(x.shape)}"
assert y.shape == (BATCH_SIZE,), f"Batch nhãn phải có shape (128,), đang là {tuple(y.shape)}"
assert x.dtype == torch.float32, "Ảnh phải là float32 (thiếu ToTensor?)"
assert -0.5 < x.mean().item() < 0.5, f"Mean của batch ({x.mean():.2f}) phải gần 0. Đã Normalize chưa?"
assert len(test_tf.transforms) == 2, "test_tf chỉ nên có ToTensor và Normalize (không augmentation)"
assert train_loader.sampler.__class__.__name__ == "RandomSampler", "train_loader phải shuffle=True"
assert val_loader.sampler.__class__.__name__ == "SequentialSampler", "val_loader không nên shuffle"
print("✅ Phần 1 đúng! x:", tuple(x.shape), "| mean ≈ %.3f, std ≈ %.3f" % (x.mean(), x.std()))

In [ ]:
# Xem thử 16 ảnh trong batch (đã viết sẵn). Ảnh đã bị augmentation nên có thể lệch/lật.
def show_batch(x, y, n=16):
    mean = torch.tensor(MEAN).view(3, 1, 1); std = torch.tensor(STD).view(3, 1, 1)
    fig, axes = plt.subplots(2, 8, figsize=(14, 4))
    for i, ax in enumerate(axes.flat[:n]):
        img = (x[i].cpu() * std + mean).clamp(0, 1).permute(1, 2, 0)  # bỏ normalize để hiển thị
        ax.imshow(img); ax.set_title(CLASSES[y[i]], fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()

show_batch(x, y)

## Phần 2 — Model CNN

Một **block** gồm 2 lớp tích chập rồi giảm kích thước một nửa:

```
Conv2d(3×3, padding=1) → BatchNorm2d → ReLU → Conv2d(3×3, padding=1) → BatchNorm2d → ReLU → MaxPool2d(2)
```

- `Conv2d(in, out, 3, padding=1)`: học các bộ lọc 3×3. `padding=1` giữ nguyên chiều cao/rộng.
- `BatchNorm2d(out)`: chuẩn hóa đầu ra theo batch → train nhanh và ổn định hơn.
- `MaxPool2d(2)`: giảm 32×32 → 16×16 (lấy giá trị lớn nhất trong mỗi ô 2×2).

Model gồm 3 block, số kênh tăng dần còn kích thước ảnh giảm dần:

| | Kênh | Kích thước |
|---|---|---|
| Đầu vào | 3 | 32×32 |
| Block 1 | 32 | 16×16 |
| Block 2 | 64 | 8×8 |
| Block 3 | 128 | 4×4 |
| Flatten | 128·4·4 = 2048 | |
| Dropout(0.3) → Linear(2048, 10) | 10 lớp | |

Đầu ra là 10 số **logits** (chưa qua softmax). `CrossEntropyLoss` sẽ tự lo phần softmax.

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32),
            conv_block(32, 64),
            conv_block(64, 128),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

<details><summary>💡 Gợi ý</summary>

- `nn.Sequential(lớp1, lớp2, ...)` nối các lớp; gọi `seq(x)` là chạy lần lượt.
- Có thể lồng `nn.Sequential` vào nhau: `nn.Sequential(conv_block(3, 32), conv_block(32, 64), ...)`.
- `forward` chỉ cần 1 dòng: `return self.classifier(self.features(x))`.
</details>

In [ ]:
# ✅ Kiểm tra Phần 2
m = SimpleCNN()
out = m(torch.randn(2, 3, 32, 32))
assert out.shape == (2, 10), f"Output phải có shape (2, 10), đang là {tuple(out.shape)}"
n_params = sum(p.numel() for p in m.parameters())
n_bn = sum(isinstance(l, nn.BatchNorm2d) for l in m.modules())
assert n_bn == 6, f"Phải có 6 lớp BatchNorm2d (2 mỗi block), đang có {n_bn}"
assert 250_000 < n_params < 350_000, f"Số tham số nên khoảng 0.3M, đang là {n_params:,}"
print(f"✅ Phần 2 đúng! Model có {n_params:,} tham số")

## Phần 3 — Hàm train một epoch (trái tim của lab)

Mỗi batch đi qua **5 bước**. Học thuộc thứ tự này, bạn sẽ gặp nó trong mọi dự án PyTorch:

```
1. optimizer.zero_grad()      # xóa gradient cũ (PyTorch CỘNG DỒN gradient nếu không xóa)
2. logits = model(x)          # forward: tính dự đoán
3. loss = criterion(logits, y)# so dự đoán với nhãn thật
4. loss.backward()            # backward: tính gradient của loss theo từng tham số
5. optimizer.step()           # cập nhật tham số theo gradient
```

Trước vòng lặp gọi `model.train()` để bật Dropout và cho BatchNorm cập nhật thống kê.

Ngoài ra ta cần **thống kê** để in ra:
- `loss.item()` là loss **trung bình** của batch → nhân với số ảnh trong batch rồi cộng dồn, cuối cùng chia tổng số ảnh.
- Số dự đoán đúng: `(logits.argmax(dim=1) == y).sum().item()`.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total += bs
    return total_loss / total, total_correct / total

### Mẹo debug quan trọng: cho model "học thuộc" 1 batch
Trước khi train thật (tốn thời gian), hãy kiểm tra model có học được **một batch nhỏ** không. Nếu code đúng, loss trên 64 ảnh cố định phải giảm gần về 0 sau vài chục bước. Nếu loss không giảm → có bug (quên `zero_grad`, quên `step`, sai nhãn...). Kỹ sư ML có kinh nghiệm luôn làm bước này.

In [ ]:
# ✅ Kiểm tra Phần 3 — overfit 1 batch
torch.manual_seed(0)
xb, yb = next(iter(val_loader))           # val_loader không augmentation → batch cố định
tiny = [(xb[:64], yb[:64])]               # "loader" chỉ có đúng 1 batch
m = SimpleCNN().to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()
losses = []
for step in range(60):
    l, a = train_one_epoch(m, tiny, opt, crit)
    losses.append(l)
print(f"loss bước đầu: {losses[0]:.3f} → bước cuối: {losses[-1]:.3f}, acc cuối: {a:.0%}")
assert losses[-1] < 0.1 * losses[0], "Loss không giảm đủ. Kiểm tra lại 5 bước (thứ tự, có quên zero_grad/step không?)"
assert a > 0.95, "Model phải học thuộc được 64 ảnh (acc > 95%)"
print("✅ Phần 3 đúng! Model học được → training loop hoạt động")

## Phần 4 — Hàm đánh giá

Giống `train_one_epoch` nhưng **không cập nhật model**:
- `model.eval()`: tắt Dropout, BatchNorm dùng thống kê đã học (không dùng thống kê của batch hiện tại).
- `with torch.no_grad():` không lưu đồ thị tính toán → nhanh hơn, tốn ít bộ nhớ hơn.
- Không có `zero_grad`, `backward`, `step`.

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            bs = y.size(0)
            total_loss += loss.item() * bs
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total += bs
    return total_loss / total, total_correct / total

In [ ]:
# ✅ Kiểm tra Phần 4
torch.manual_seed(0)
m = SimpleCNN().to(device)
before = [p.detach().clone() for p in m.parameters()]
vl, va = evaluate(m, val_loader, nn.CrossEntropyLoss())
assert all(torch.equal(a, b) for a, b in zip(before, m.parameters())), "evaluate không được thay đổi tham số model!"
assert not m.training, "Sau evaluate, model phải ở chế độ eval (quên model.eval()?)"
assert 0.03 < va < 0.25, f"Model chưa train phải đoán gần ngẫu nhiên (~10%), đang là {va:.1%}"
assert 1.8 < vl < 3.0, f"Loss ban đầu nên gần ln(10) ≈ 2.30, đang là {vl:.2f}"
print(f"✅ Phần 4 đúng! Model chưa train: val loss {vl:.3f}, val acc {va:.1%} (≈ đoán bừa 10%)")

## Phần 5 — Checkpoint

Train trên Colab có thể bị ngắt bất kỳ lúc nào. Checkpoint lưu **đủ thông tin để train tiếp** như chưa từng dừng:

| Khóa | Vì sao cần |
|---|---|
| `"model"` | `model.state_dict()`: trọng số |
| `"optimizer"` | `optimizer.state_dict()`: Adam lưu momentum riêng cho từng tham số, thiếu nó train tiếp sẽ "giật" |
| `"scheduler"` | `scheduler.state_dict()`: để learning rate tiếp tục đúng lịch |
| `"epoch"` | epoch vừa xong |
| `"best_acc"` | val acc tốt nhất đến giờ |

Dùng `torch.save(dict, path)` để lưu và `torch.load(path, map_location=device)` để nạp, sau đó `model.load_state_dict(...)`.

In [ ]:
def save_checkpoint(path, model, optimizer, scheduler, epoch, best_acc):
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "best_acc": best_acc,
    }, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])
    if optimizer is not None:
        optimizer.load_state_dict(ckpt["optimizer"])
    if scheduler is not None:
        scheduler.load_state_dict(ckpt["scheduler"])
    return ckpt["epoch"], ckpt["best_acc"]

In [ ]:
# ✅ Kiểm tra Phần 5
torch.manual_seed(0)
m1 = SimpleCNN().to(device)
o1 = torch.optim.AdamW(m1.parameters(), lr=1e-3)
s1 = torch.optim.lr_scheduler.CosineAnnealingLR(o1, T_max=10)
train_one_epoch(m1, tiny, o1, nn.CrossEntropyLoss()); s1.step()
save_checkpoint("test_ckpt.pt", m1, o1, s1, epoch=3, best_acc=0.5)

m2 = SimpleCNN().to(device)
o2 = torch.optim.AdamW(m2.parameters(), lr=1e-3)
s2 = torch.optim.lr_scheduler.CosineAnnealingLR(o2, T_max=10)
ep, best = load_checkpoint("test_ckpt.pt", m2, o2, s2)
assert (ep, best) == (3, 0.5), f"Phải trả về (3, 0.5), đang là {(ep, best)}"
assert all(torch.equal(a, b) for a, b in zip(m1.state_dict().values(), m2.state_dict().values())), "Trọng số nạp vào không khớp"
assert len(o2.state_dict()["state"]) > 0, "Optimizer state chưa được nạp"
assert s2.last_epoch == s1.last_epoch, "Scheduler state chưa được nạp"
os.remove("test_ckpt.pt")
print("✅ Phần 5 đúng! Lưu/nạp checkpoint hoạt động")

## Phần 6 — Ghép lại: vòng lặp train đầy đủ

Mỗi epoch:
1. `train_one_epoch` → train loss, train acc
2. `evaluate` trên `val_loader` → val loss, val acc
3. `scheduler.step()`: giảm learning rate dần theo đường cosine
4. Lưu vào `history` để vẽ biểu đồ
5. Luôn lưu `last.pt`; nếu val acc tốt hơn `best_acc` thì cập nhật và lưu thêm `best.pt`
6. In một dòng tiến độ

**Train tiếp:** nếu `RESUME = True` và có `last.pt`, nạp nó và bắt đầu từ `epoch + 1`.

Trên GPU T4 mỗi epoch khoảng 20–40 giây.

In [ ]:
EPOCHS = 15
RESUME = False
CKPT_DIR = "checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

torch.manual_seed(42)
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

start_epoch, best_acc = 1, 0.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

last_path = os.path.join(CKPT_DIR, "last.pt")
best_path = os.path.join(CKPT_DIR, "best.pt")
if RESUME and os.path.exists(last_path):
    last_epoch, best_acc = load_checkpoint(last_path, model, optimizer, scheduler)
    start_epoch = last_epoch + 1
    print(f"↩️ Train tiếp từ epoch {start_epoch} (best val acc hiện tại {best_acc:.2%})")

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    va_loss, va_acc = evaluate(model, val_loader, criterion)
    scheduler.step()

    for k, v in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
        history[k].append(v)

    is_best = va_acc > best_acc
    if is_best:
        best_acc = va_acc
        save_checkpoint(best_path, model, optimizer, scheduler, epoch, best_acc)
    save_checkpoint(last_path, model, optimizer, scheduler, epoch, best_acc)

    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {tr_loss:.3f} acc {tr_acc:.2%} | "
          f"val loss {va_loss:.3f} acc {va_acc:.2%} | {time.time() - t0:.0f}s" + ("  ⭐ best" if is_best else ""))

<details><summary>💡 Gợi ý TODO 6.1</summary>

```python
last_epoch, best_acc = load_checkpoint(last_path, model, optimizer, scheduler)
start_epoch = last_epoch + 1
```
</details>

<details><summary>💡 Gợi ý TODO 6.2</summary>

```python
tr_loss, tr_acc = train_one_epoch(...)
va_loss, va_acc = evaluate(...)
scheduler.step()
history["train_loss"].append(tr_loss)   # ... tương tự 3 khóa còn lại
if va_acc > best_acc:
    best_acc = va_acc
    save_checkpoint(best_path, ...)
save_checkpoint(last_path, ...)
print(f"Epoch {epoch} | ...")
```
</details>

In [ ]:
# Vẽ đường học (đã viết sẵn)
if history["val_acc"]:
    ep = range(len(history["val_acc"]))
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot(ep, history["train_loss"], label="train"); a1.plot(ep, history["val_loss"], label="val")
    a1.set_title("Loss"); a1.set_xlabel("epoch"); a1.legend()
    a2.plot(ep, history["train_acc"], label="train"); a2.plot(ep, history["val_acc"], label="val")
    a2.axhline(0.70, ls="--", c="gray", label="mục tiêu 70%")
    a2.set_title("Accuracy"); a2.set_xlabel("epoch"); a2.legend()
    plt.tight_layout(); plt.show()

**Đọc biểu đồ:**
- Hai đường val và train đi sát nhau → model chưa overfit, có thể train lâu hơn hoặc dùng model lớn hơn.
- Train acc tiếp tục tăng mà val acc đi ngang hoặc giảm → **overfitting**. Augmentation, Dropout và weight decay là 3 thứ ta đã dùng để chống nó.
- Thường ở lab này train acc thấp hơn val acc ở vài epoch đầu. Không phải lỗi: train bị augmentation và Dropout nên "khó" hơn.

## Phần 7 — Kiểm tra cuối: đánh giá trên tập test và thử train tiếp

### 7a. Test
Nạp `best.pt` vào **một model mới** và đánh giá trên `test_loader`. Đây là con số bạn ghi vào README.

In [ ]:
final_model = SimpleCNN().to(device)
best_epoch, _ = load_checkpoint(best_path, final_model)
test_loss, test_acc = evaluate(final_model, test_loader, criterion)
print(f"Best checkpoint từ epoch {best_epoch}")

In [ ]:
# ✅ Kiểm tra cuối
assert test_acc is not None, "Chưa làm TODO 7"
print(f"Test loss {test_loss:.3f} | Test accuracy {test_acc:.2%}")
assert test_acc >= 0.70, f"Chưa đạt 70% ({test_acc:.1%}). Thử tăng EPOCHS hoặc kiểm tra lại augmentation/normalize."
print("🎉 Hoàn thành Lab 1! Test accuracy ≥ 70%")

### 7b. Thử train tiếp từ checkpoint
1. Ở ô Phần 6, đổi `EPOCHS = 20` và `RESUME = True`.
2. Chạy lại ô đó. Bạn phải thấy dòng `↩️ Train tiếp từ epoch 16`, và loss của epoch 16 **liền mạch** với epoch 15 (không vọt lên).

> Lưu ý: khi đổi `EPOCHS`, scheduler cosine đã được tạo với `T_max=15` và nạp lại trạng thái cũ, nên learning rate sau epoch 15 sẽ không theo đúng đường cong mới. Đó là một câu hỏi tự suy nghĩ bên dưới.

---

## Phần 8 — Câu hỏi tự kiểm tra
Trả lời bằng lời của bạn (gửi cho Claude để chấm):

1. Chuyện gì xảy ra nếu bỏ `optimizer.zero_grad()`? Thử xóa dòng đó, chạy ô kiểm tra Phần 3 và mô tả kết quả.
2. Vì sao tập val dùng `test_tf` chứ không dùng `train_tf`?
3. Quên `model.eval()` khi đánh giá thì kết quả sai ở đâu? (gợi ý: 2 lớp nào hoạt động khác nhau giữa train và eval)
4. Vì sao phải lưu cả state của optimizer trong checkpoint?
5. Ở Phần 7b, làm sao để train tiếp mà learning rate vẫn đúng lịch cosine cho 20 epoch?

## Thử thách mở rộng (tùy chọn)
- **A.** Thêm *mixed precision* (`torch.autocast` + `torch.amp.GradScaler`) và so sánh thời gian mỗi epoch.
- **B.** Thay AdamW bằng SGD (momentum 0.9, lr 0.1). Cái nào tốt hơn sau 15 epoch?
- **C.** Vẽ confusion matrix trên tập test. Lớp nào hay bị nhầm với nhau nhất?
- **D.** Tăng số kênh lên (64, 128, 256), train 30 epoch. Bạn đạt được bao nhiêu? (≈ 90% là có thể)

## Nộp bài lên GitHub
Tạo repo `ai-engineer-labs`, thư mục `lab01-training-loop/` gồm:
- notebook này (đã chạy, giữ output)
- `README.md`: mục tiêu, kiến trúc model, kết quả test accuracy, biểu đồ, 2–3 điều bạn rút ra